# Challenge:

- Build a synthetic dataset generator, write models that can generate datasets.
- Use a variety of models and prompts for diverse output, try quantizing and not quantizing, try different shapes and sizes.
- Create a Gradio UI for your product

## Idea

- Generate sample data for use when developing a REST API.

## ⚠️ Before you run this notebook

**GPU required.** This notebook uses 4-bit quantized models via BitsAndBytes and requires an NVIDIA GPU with CUDA support. CPU inference is not supported.

**Disk space.** Models are downloaded from Hugging Face on first run and cached locally (`~/.cache/huggingface/hub`). Downloads are a one-time cost per model:

| Model | Download size (approx) |
|-------|----------------------|
| Llama 3.1 8B Instruct | ~16 GB |
| Qwen 2.5 Coder 7B Instruct | ~15 GB |
| Phi 4 Mini Instruct | ~5 GB |

**Total: ~36 GB** on first run. Subsequent runs load from cache.

**HF Token.** Llama requires a Hugging Face account with access approved at [meta-llama/Meta-Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct). Set `HF_TOKEN` in your `.env` file.

---

## Install order matters

Install CUDA-enabled PyTorch first, then other packages with `--no-deps` to prevent them pulling in a CPU-only torch build:

```bash
uv pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
uv pip install --no-deps accelerate bitsandbytes transformers==4.57.6 huggingface_hub[hf_xet]
```

On a local environment these installs are persistent — you only need to run them once.


In [ ]:
import torch, accelerate, bitsandbytes, transformers

print(f"torch=={torch.__version__}")
print(f"accelerate=={accelerate.__version__}")
print(f"bitsandbytes=={bitsandbytes.__version__}")
print(f"transformers=={transformers.__version__}")

In [ ]:
# Imports

from transformers import (
    pipeline,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
import os
from dotenv import load_dotenv
from openai import OpenAI
from huggingface_hub import login

load_dotenv(override=True)

In [ ]:
# Constants

LLAMA_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
QWEN_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
PHI_MODEL = "microsoft/Phi-4-mini-instruct"
OPENAPI_FILE = "storefront-sample.json"

In [ ]:
# Huggingface login

login(token=os.getenv("HF_TOKEN"), add_to_git_credential=True)

In [ ]:
# in case $ref is used in the schema - resolve_ref


def resolve_ref(spec, ref):
    path = ref.replace("#/", "").split("/")
    obj = spec
    for p in path:
        obj = obj[p]
    return obj

In [ ]:
# Extraction function

import json


def extract_schema(openapi, path, method):
    method = method.lower()
    endpoint = openapi["paths"][path][method]
    schema = endpoint["requestBody"]["content"]["application/json"]["schema"]
    return schema


# Note: at this stage we intentionally extract only ONE endpoint.
# The extraction helper works for any path/method; the one-endpoint choice keeps the notebook minimal.
ENDPOINT_PATH = "/cart/items"
ENDPOINT_METHOD = "post"

with open(OPENAPI_FILE) as f:
    spec = json.load(f)

schema = extract_schema(spec, ENDPOINT_PATH, ENDPOINT_METHOD)

if "$ref" in schema:
    schema = resolve_ref(spec, schema["$ref"])

print(json.dumps(schema, indent=2))

In [ ]:
# Kept as minimal as possible by intent

user_prompt = f"""Generate realistic JSON objects following this schema:


{schema}
"""


system_prompt = """


You excel at generating realistic JSON objects following a given schema.
"""


messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

In [ ]:
import torch

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)


def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name, device_map="auto", quantization_config=quant_config
    )
    return model, tokenizer


def generate_json(model, tokenizer, messages, max_new_tokens=1000):
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", return_dict=True, add_generation_prompt=True
    ).to("cuda")

    print(f"  Input tokens: {inputs['input_ids'].shape[1]}")
    print(f"  Chat template found: {tokenizer.chat_template is not None}")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[1] :]
    print(f"  Output tokens: {len(new_tokens)}")
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
import re
from jsonschema import validate, ValidationError


def extract_json(text):
    """Pull the first complete JSON object or array out of model output."""
    if text is None:
        return None

    # Try ```json ... ``` code blocks first
    code_block = re.search(r"```(?:json)?\s*(\{[\s\S]*?\}|\[[\s\S]*?\])\s*```", text)
    if code_block:
        return code_block.group(1).strip()

    # Fall back to matching outermost braces/brackets
    for start, ch in enumerate(text):
        if ch not in "{[":
            continue
        close = "}" if ch == "{" else "]"
        depth = 0
        for end in range(start, len(text)):
            if text[end] == ch:
                depth += 1
            elif text[end] == close:
                depth -= 1
                if depth == 0:
                    return text[start : end + 1]
    return None


def validate_json(text):
    """Parse JSON string, return (parsed_object, is_valid) tuple."""
    try:
        return json.loads(text), True
    except (json.JSONDecodeError, TypeError):
        return None, False


def validate_schema(data, target_schema):
    """Check if data conforms to the target schema. Returns (conforms, error_message)."""
    items = data if isinstance(data, list) else [data]
    errors = []
    for i, item in enumerate(items):
        try:
            validate(instance=item, schema=target_schema)
        except ValidationError as e:
            errors.append(f"Item {i}: {e.message}")
    if errors:
        return False, "; ".join(errors)
    return True, None

In [ ]:
import gc

MODELS = {
    "Llama 3.1 8B": LLAMA_MODEL,
    "Qwen 2.5 Coder 7B": QWEN_MODEL,
    "Phi 4 Mini": PHI_MODEL,
}

results = []

for label, model_name in MODELS.items():
    print(f"\n{'=' * 60}")
    print(f"  {label} ({model_name})")
    print(f"{'=' * 60}")

    try:
        model, tokenizer = load_model(model_name)
        raw_output = generate_json(model, tokenizer, messages)

        extracted = extract_json(raw_output)
        parsed, is_valid = validate_json(extracted)
        conforms, schema_errors = (
            validate_schema(parsed, schema) if is_valid else (False, "No valid JSON")
        )

        results.append(
            {
                "model": label,
                "valid_json": is_valid,
                "schema_match": conforms,
                "schema_errors": schema_errors,
                "raw_output": raw_output,
                "extracted": extracted,
                "parsed": parsed,
            }
        )

        print(f"\nRaw output:\n{raw_output}")
        print(f"\nExtracted JSON:\n{extracted}")
        print(f"\nValid JSON: {is_valid}")
        print(f"Schema match: {conforms}")
        if schema_errors:
            print(f"Schema errors: {schema_errors}")

        del model, tokenizer
        torch.cuda.empty_cache()
        gc.collect()

    except Exception as e:
        print(f"\nError: {e}")
        results.append(
            {
                "model": label,
                "valid_json": False,
                "raw_output": None,
                "extracted": None,
                "parsed": None,
                "error": str(e),
                "schema_match": False,
                "schema_errors": str(e),
            }
        )

In [ ]:
print(f"\n{'=' * 60}")
print("  Model Comparison Summary")
print(f"{'=' * 60}\n")
print(f"  {'Model':<25} {'Valid JSON':<14} {'Schema Match'}")
print(f"  {'-' * 53}")
for r in results:
    print(f"  {r['model']:<25} {str(r['valid_json']):<14} {r['schema_match']}")
    if r.get("schema_errors"):
        print(f"    -> {r['schema_errors']}")